# Mitigating VQE errors on the H₂ molecule with EMRG

The Variational Quantum Eigensolver (VQE) estimates molecular ground-state energies by
optimizing a parameterized quantum circuit. On current NISQ hardware, gate errors
distort the energy landscape and shift the optimized energy away from the true ground
state. For small molecules like H₂, the circuit is shallow enough that error mitigation
can recover most of the lost accuracy.

EMRG analyzes a quantum circuit and generates a ready-to-run error mitigation recipe
using Mitiq. It selects the technique (ZNE or PEC), the extrapolation factory, and the
noise scaling method based on circuit characteristics. No manual configuration required.

This notebook builds a 4-qubit H₂ VQE ansatz, runs it through EMRG's analysis and
code generation pipeline, and compares ZNE and PEC recipes.

## Setup

In [ ]:
# pip install emrg qiskit

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

import emrg

## Build the H₂ VQE ansatz

We use a hardware-efficient ansatz with 4 qubits, 2 layers of RY rotations and CNOT
entanglement. This is a standard structure for H₂ ground-state estimation in the
STO-3G basis.

This ansatz structure follows patterns identified in autonomous VQE architecture
search ([github.com/FedorShind/autoresearch-qc](https://github.com/FedorShind/autoresearch-qc)).

The parameters are bound to fixed values for this demonstration. In a real VQE loop,
a classical optimizer would update them.

In [ ]:
from qiskit.circuit import Parameter

n_qubits = 4
n_layers = 2

qc = QuantumCircuit(n_qubits, n_qubits)

params = []
for layer in range(n_layers):
    # RY rotation on each qubit
    for q in range(n_qubits):
        p = Parameter(f"theta_{layer}_{q}")
        params.append(p)
        qc.ry(p, q)
    # CNOT entanglement (linear chain)
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)

qc.measure(range(n_qubits), range(n_qubits))

# Bind parameters to example values
param_values = {p: np.random.uniform(0, 2 * np.pi) for p in params}
qc = qc.assign_parameters(param_values)

print(f"Qubits: {qc.num_qubits}")
print(f"Depth:  {qc.depth()}")
print(qc.draw())

## Analyze the circuit with EMRG

`analyze_circuit` extracts features used by the heuristic engine to select a
mitigation strategy. No simulation is performed -- this is purely static analysis.

In [ ]:
features = emrg.analyze_circuit(qc)

print(f"Qubits:              {features.num_qubits}")
print(f"Depth:               {features.depth}")
print(f"Total gates:         {features.total_gate_count}")
print(f"Multi-qubit gates:   {features.multi_qubit_gate_count}")
print(f"Single-qubit gates:  {features.single_qubit_gate_count}")
print(f"Noise estimate:      {features.estimated_noise_factor}")
print(f"Noise category:      {features.noise_category}")
print(f"PEC overhead est:    {features.pec_overhead_estimate:.2f}")
print(f"Layer heterogeneity: {features.layer_heterogeneity:.4f}")

Key fields:

- **depth** -- the longest path of dependent operations. Determines which ZNE factory
  is appropriate.
- **multi_qubit_gate_count** -- two-qubit gates dominate the error budget on current
  hardware (typical error rates ~0.5-1% vs ~0.01% for single-qubit gates).
- **estimated_noise_factor** -- a heuristic proxy: `multi_qubit_gates * 0.01 +
  single_qubit_gates * 0.001`. Not a rigorous noise model, but sufficient for
  strategy selection.
- **pec_overhead_estimate** -- `exp(2 * noise_factor * multi_qubit_gates)`. Values
  above ~1000 make PEC impractical.
- **layer_heterogeneity** -- ratio of max to min multi-qubit gate counts across
  layers. High values favour layerwise noise scaling.

## Generate a ZNE recipe

With `explain=True`, the generated code includes rationale comments explaining
why each configuration was chosen.

In [ ]:
result = emrg.generate_recipe(qc, explain=True)

print(result.code)

The generated code is a complete Python script. It imports the selected Mitiq factory
and scaling function, constructs the factory with the recommended scale factors, and
calls `execute_with_zne`. The executor function is a placeholder -- replace it with
your actual backend (Aer simulator, IBM Quantum hardware, etc.).

EMRG does not execute the mitigated circuit. It generates the recipe. You run it.

The rationale lines explain the decision:

In [ ]:
for line in result.rationale:
    print(f"  - {line}")

## Generate a PEC recipe for comparison

PEC (Probabilistic Error Cancellation) requires a noise model. Setting
`noise_model_available=True` tells EMRG that a depolarizing noise model can be
supplied, enabling PEC consideration.

In [ ]:
pec_result = emrg.generate_recipe(
    qc, noise_model_available=True, explain=True
)

print(f"Technique: {pec_result.recipe.technique}")
print(f"Factory:   {pec_result.recipe.factory_name or 'N/A (PEC)'}")
print()
print(pec_result.code)

If EMRG selected PEC, it is because the circuit is shallow enough and the estimated
overhead is below the 1000x threshold. PEC provides unbiased error cancellation but
at a higher sampling cost than ZNE.

If the overhead is too high, EMRG falls back to ZNE even when a noise model is
available. You can also force a technique with `technique="pec"` or `technique="zne"`:

In [ ]:
forced_pec = emrg.generate_recipe(
    qc, technique="pec", noise_model_available=True
)
print(f"Forced technique: {forced_pec.recipe.technique}")
print(f"Overhead estimate: {forced_pec.features.pec_overhead_estimate:.2f}")

**When to choose ZNE vs PEC:**

| | ZNE | PEC |
|---|---|---|
| Requires noise model | No | Yes |
| Bias | Biased (extrapolation) | Unbiased |
| Sampling overhead | Low (3-5x) | High (exponential in noise) |
| Best for | General use, deeper circuits | Shallow circuits with known noise |

## Summary

EMRG automated the following steps:

1. Extracted circuit features (depth, gate counts, noise estimate, layer heterogeneity)
2. Selected a mitigation technique and configuration based on heuristic rules
3. Generated a complete, runnable Python script with Mitiq imports and inline rationale

Links:
- [EMRG on GitHub](https://github.com/FedorShind/EMRG)
- [EMRG on PyPI](https://pypi.org/project/emrg/)
- [Mitiq documentation](https://mitiq.readthedocs.io/)